# APMC catalog -- initial extraction (Opus vision)

This notebook produces the initial 3-column CSV (`Listing`, `Page`, `Images Above`)
for an OCR'd ASCC catalog by sending each split half-page PNG to Claude Opus
and asking for the listings in top-to-bottom order. The output is the upstream
input that [apmc_page_check.ipynb](apmc_page_check.ipynb) verifies and corrects.

Inputs:  `wip/in/<BASENAME>/page-NNNN-{L,R}.png`
Output:  `wip/out/<BASENAME>_extracted.csv`
Cache:   `wip/cache/<BASENAME>_extract.json`

Stages:

1. Setup and config.
2. Discover the half-page PNGs.
3. System prompt (frozen string, prompt-cached).
4. Vision call function.
5. Cache + loop -- one Opus call per half-page.
6. Assemble the DataFrame.
7. Write the CSV.
8. Verification notes.


In [1]:
# 1. Setup and config.
import base64, json, os, re
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import anthropic

# Repo-root .env (notebook cwd is tools/).
load_dotenv(Path("..") / ".env")

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

BASENAME   = "VA_ASCC_CTLG"
IMAGES_DIR = Path(f"./wip/in/{BASENAME}")
OUTPUT_CSV = Path(f"./wip/out/{BASENAME}_extracted.csv")
CACHE_FILE = Path(f"./wip/cache/{BASENAME}_extract.json")
CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

EXTRACT_MODEL          = "claude-opus-4-7"
EXTRACT_PROMPT_VERSION = "v1"
EXTRACT_MAX_TOKENS     = 8192   # each half-page may carry ~30+ listings
FORCE_REFRESH          = False

assert IMAGES_DIR.is_dir(), f"missing images dir: {IMAGES_DIR}"
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set"
client = anthropic.Anthropic()
print(f"model: {EXTRACT_MODEL}  prompt_version: {EXTRACT_PROMPT_VERSION}")
print(f"images: {IMAGES_DIR}")
print(f"output: {OUTPUT_CSV}")
print(f"cache:  {CACHE_FILE}")


model: claude-opus-4-7  prompt_version: v1
images: wip/in/VA_ASCC_CTLG
output: wip/out/VA_ASCC_CTLG_extracted.csv
cache:  wip/cache/VA_ASCC_CTLG_extract.json


In [2]:
# 2. Discover the half-page PNGs and sort into reading order.
NAME_RE = re.compile(r"^page-(\d{4})-([LR])\.png$")

halves = []
for p in sorted(IMAGES_DIR.glob("page-*-?.png")):
    m = NAME_RE.match(p.name)
    if not m:
        continue
    halves.append((int(m.group(1)), m.group(2), p))

# Reading order: ascending page, then L before R.
halves.sort(key=lambda t: (t[0], 0 if t[1] == "L" else 1))

pages = sorted({h[0] for h in halves})
assert halves, f"no PNGs matched in {IMAGES_DIR}"
print(f"found {len(halves)} half-pages across pages {pages[0]}-{pages[-1]} "
      f"({len(pages)} pages total)")


found 34 half-pages across pages 419-435 (17 pages total)


## 3. System prompt

Frozen string sent with `cache_control: ephemeral` so prompt caching hits on
calls within the 5-minute TTL. The prompt instructs Opus to return strict JSON
only and shows the schema by example.


In [3]:
# 3. System prompt for the extraction model.
EXTRACT_SYSTEM_PROMPT = """You are reading half-page scans of an old American Stampless Cover (ASCC) catalog.

Each catalog page is a single column of catalog entries, read top-to-bottom.
Pages are split into two halves: -L (top) and -R (bottom). The image you
receive is ONE half-page and contains only the entries that appear in that
half.

Your job: emit every catalog listing visible on this half-page, in top-to-
bottom order, as strict JSON. For each listing, also count how many distinct
stamp / postmark / handstamp images appear in the vertical gap directly
ABOVE that listing's text on this half-page.

What is a listing
-----------------

A listing is one catalog entry. It typically follows this shape:

    TOWN(date;manuscript-or-printed;color) ......... value

Examples drawn verbatim from this catalog:

    Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00
    Fredg.(Aug.2,1772;Ms;Black) .. 1500.00
    (L)(Sept.15,1774) . . .. 1000.00

Continuation listings start with the literal word Same followed by an open
paren -- e.g. Same(Aug.2,1772;Ms;Black) .. 1500.00 -- and are SEPARATE
listings. Emit each Same(...) line as its own listing object. Preserve any
leading whitespace / indentation in the listing text.

Some entries wrap onto a second line (a continuation that begins with an
open paren but not the word Same). Treat that as part of the SAME listing
and join the wrapped text into one string.

NOT listings (skip them):
- printed page numbers
- section headings (e.g. "VIRGINIA")
- running heads / column headers
- decorative ornaments
- the stamp / postmark images themselves

Listing text fidelity
---------------------

Reproduce the listing text EXACTLY as printed. Preserve case, spacing,
punctuation, semicolons, slashes, parentheses, leader dots, and the price
value at the end. Do NOT parse or split the entry into sub-fields. Do NOT
normalize spelling, expand abbreviations, or "clean up" the leader dots.

ASCII only. Use straight " and ' (not curly), use - or -- (not en-dash or
em-dash), use three dots ... (not the ellipsis character), no accented
letters, no Unicode bullets or arrows.

If a listing is fully illegible, OMIT it. Do not fabricate text. If only
the price is illegible, emit the listing with whatever text you can read
faithfully and use ? in place of the unreadable glyphs.

Counting images_above
---------------------

For each listing, count the distinct stamp / postmark / handstamp image
reproductions that sit in the vertical gap directly above that listing's
text on the half-page, between the previous listing's text on the same
half and this one. Most listings have 0.

For the topmost listing on the half-page, count images between the top of
the half (or the running head / section heading, if present) and the
listing's text. The bottom half (-R) starts where the top half (-L) ended,
so do NOT carry an image count across the L/R seam -- just count what is
above the topmost listing within the half-page you were given.

Rules:
- ONE stamp image is ONE image. A small adjacent date or rate annotation
  printed next to the stamp is part of the same image, NOT a second image.
- Two genuinely separate stamp reproductions placed close together are
  TWO images.
- If unsure between 1 and 2, prefer 1.

Output format
-------------

Return STRICT JSON only -- no surrounding prose, no markdown fences, no
trailing commentary. The shape:

    {"listings": [
      {"listing": "Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00", "images_above": 0},
      {"listing": "Same(Aug.2,1772;Ms;Black) .. 1500.00", "images_above": 1}
    ]}

If the half-page contains zero listings, return {"listings": []}.
"""

print(f"system prompt length: {len(EXTRACT_SYSTEM_PROMPT)} chars")


system prompt length: 3662 chars


In [4]:
# 4. Extraction call. Returns the parsed listings array from the response.

def extract_listings(image_path: Path, half: str, page: int) -> list[dict]:
    assert half in ("L", "R")
    side = "TOP half (-L)" if half == "L" else "BOTTOM half (-R)"
    user_prompt = (
        f"This image is the {side} of ASCC catalog page {page}. "
        f"Return strict JSON of the form "
        f'{{"listings": [{{"listing": "...", "images_above": 0}}, ...]}}.'
    )
    msg = client.messages.create(
        model=EXTRACT_MODEL,
        max_tokens=EXTRACT_MAX_TOKENS,
        system=[{
            "type": "text",
            "text": EXTRACT_SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": base64.standard_b64encode(image_path.read_bytes()).decode(),
            }},
            {"type": "text", "text": user_prompt},
        ]}],
    )
    text = next(b.text for b in msg.content if b.type == "text").strip()
    parsed = json.loads(text)
    assert isinstance(parsed, dict), f"top-level not dict: {type(parsed)}"
    listings = parsed.get("listings")
    assert isinstance(listings, list), f"listings not list: {type(listings)}"
    for i, r in enumerate(listings):
        assert isinstance(r, dict), f"listing[{i}] not dict"
        assert isinstance(r.get("listing"), str), f"listing[{i}].listing not str"
        ia = r.get("images_above")
        assert isinstance(ia, int) and ia >= 0, f"listing[{i}].images_above bad: {ia!r}"
    return listings


## 5. Cache and loop

Disk cache is invalidated when `EXTRACT_MODEL` or `EXTRACT_PROMPT_VERSION`
changes. Cache is saved after each successful call so a crash mid-loop only
loses the in-flight call. Set `FORCE_REFRESH = True` in the config cell to
re-query everything.


In [5]:
# 5. Cache + loop. One Opus call per half-page.

def load_cache(path: Path) -> dict:
    if not path.exists():
        return {"model": EXTRACT_MODEL, "prompt_version": EXTRACT_PROMPT_VERSION, "responses": {}}
    cache = json.loads(path.read_text())
    if cache.get("model") != EXTRACT_MODEL or cache.get("prompt_version") != EXTRACT_PROMPT_VERSION:
        print(f"cache invalidated (was model={cache.get('model')!r}, prompt={cache.get('prompt_version')!r})")
        return {"model": EXTRACT_MODEL, "prompt_version": EXTRACT_PROMPT_VERSION, "responses": {}}
    return cache

def save_cache(path: Path, cache: dict) -> None:
    path.write_text(json.dumps(cache, indent=2))

cache = load_cache(CACHE_FILE)
responses = cache["responses"]

calls_made = 0
for page, half, img_path in halves:
    key = f"page-{page:04d}-{half}"
    if key in responses and not FORCE_REFRESH:
        continue
    if not img_path.exists():
        print(f"missing image, skipping: {img_path}")
        continue
    try:
        listings = extract_listings(img_path, half, page)
    except Exception as e:
        print(f"  {key}: FAILED ({type(e).__name__}: {e})")
        save_cache(CACHE_FILE, cache)
        raise
    responses[key] = {"listings": listings}
    save_cache(CACHE_FILE, cache)
    calls_made += 1
    print(f"  {key}: {len(listings):3d} listings")

print()
print(f"calls made: {calls_made}")
print(f"cached responses: {len(responses)}")


  page-0419-L:  15 listings
  page-0419-R:  25 listings
  page-0420-L:  29 listings
  page-0420-R:  24 listings
  page-0421-L:  20 listings
  page-0421-R:  36 listings
  page-0422-L:  34 listings
  page-0422-R:  27 listings
  page-0423-L:  25 listings
  page-0423-R:  45 listings
  page-0424-L:  34 listings
  page-0424-R:  34 listings
  page-0425-L:  27 listings
  page-0425-R:  39 listings
  page-0426-L:  33 listings
  page-0426-R:  31 listings
  page-0427-L:  41 listings
  page-0427-R:  42 listings
  page-0428-L:  23 listings
  page-0428-R:  25 listings
  page-0429-L:  48 listings
  page-0429-R:  38 listings
  page-0430-L:  41 listings
  page-0430-R:  35 listings
  page-0431-L:  43 listings
  page-0431-R:  85 listings
  page-0432-L:  84 listings
  page-0432-R:  83 listings
  page-0433-L:  85 listings
  page-0433-R:  85 listings
  page-0434-L:  84 listings
  page-0434-R:  85 listings
  page-0435-L:  84 listings
  page-0435-R:  85 listings

calls made: 34
cached responses: 34


## 6. Assemble the DataFrame

Walk the half-page list in catalog reading order, pull each half's listings
from the cache, and append rows. The CSV's `Page` column is filled in here
from the parsed filename `NNNN` (the same value for every row produced from
that half-page).


In [6]:
# 6. Assemble DataFrame from the cached responses.
rows = []
for page, half, _ in halves:
    key = f"page-{page:04d}-{half}"
    entry = responses.get(key)
    if entry is None:
        print(f"WARNING: no cached response for {key}; skipping")
        continue
    for r in entry["listings"]:
        rows.append({
            "Listing": r["listing"],
            "Page": int(page),
            "Images Above": int(r["images_above"]),
        })

df = pd.DataFrame(rows, columns=["Listing", "Page", "Images Above"])
print(f"rows: {len(df):,}")
print(f"pages: {df.Page.min()}-{df.Page.max()}")
print()
print("rows per page:")
print(df.groupby("Page").size().rename("rows").to_frame())
print()
print("Images Above value counts:")
print(df["Images Above"].value_counts().sort_index().rename("rows").to_frame())
df.head(5)


rows: 1,574
pages: 419-435

rows per page:
      rows
Page      
419     40
420     53
421     56
422     61
423     70
424     68
425     66
426     64
427     83
428     48
429     86
430     76
431    128
432    167
433    170
434    169
435    169

Images Above value counts:
              rows
Images Above      
0             1413
1              128
2               29
3                4


,Listing,Page,Images Above
0,"Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) ......... 1500.00",419,1
1,"(L)(Sept.15,1774) ................................ 1000.00",419,0
2,"Colchester(June 30,1774;Ms;Black) ................ 1500.00",419,1
3,"Fredericksburg(May 10,1755;Ms;Black) .............. 2000.00",419,1
4,"Fredg.(Aug.2,1772;Ms;Black) ...................... 1500.00",419,1


## 7. Write the CSV

Pandas' `to_csv` default is RFC 4180 compliant: UTF-8, no BOM, header row,
fields containing commas or quotes are double-quoted.


In [7]:
# 7. Write CSV.
df[["Listing", "Page", "Images Above"]].to_csv(OUTPUT_CSV, index=False)
print(f"wrote: {OUTPUT_CSV}")
print(f"rows:  {len(df):,}")
print(f"pages: {df.Page.min()}-{df.Page.max()}")


wrote: wip/out/VA_ASCC_CTLG_extracted.csv
rows:  1,574
pages: 419-435


## 8. Verification

* First run hits the API once per half-page. Re-running cell 5 should print
  `calls made: 0` (everything served from `wip/cache/<BASENAME>_extract.json`).
* Spot-check: open `wip/in/<BASENAME>/page-NNNN-L.png` next to
  `df.query("Page == NNNN").head(20)`. Listings should appear in the same
  top-to-bottom order, glyph-for-glyph match on the listing text, and
  `Images Above` should match the visible postmark count above each entry.
* If you want page-boundary correction next, copy the output to
  `wip/in/<BASENAME>.csv` and run [apmc_page_check.ipynb](apmc_page_check.ipynb).
